# Level 0

## Level 0 — Dataset Collection
✔ Choose and collect your dataset

✔ Website/source used

Source: Books to Scrape
✔ URL or list of URLs

Your code stores each book's URL.
✔ Type of data collected

HTML pages
Product records
Plain text (descriptions)
✔ Structure of the data

Your JSON structure is something like:

{
  "title": "...",
  "price": "...",
  "description": "...",
  "url": "..."
}
✔ Number of records collected
Approximately 1000 book records (if you scrape the whole site).
✔ Collection method
✔ Cleaning and preprocessing

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time

In [2]:
BASE_URL = "http://books.toscrape.com/catalogue/page-{}.html"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

books = []

page = 1

while True:

    print(f"Scraping Page {page}")

    response = requests.get(BASE_URL.format(page), headers=HEADERS)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.find_all("article", class_="product_pod")

    if len(products) == 0:
        break

    for product in products:

        title = product.h3.a["title"]

        price = product.find("p", class_="price_color").text

        rating = product.p["class"][1]

        relative_url = product.h3.a["href"]

        book_url = "http://books.toscrape.com/catalogue/" + relative_url

        # Open Book Page
        details = requests.get(book_url, headers=HEADERS)

        detail_soup = BeautifulSoup(details.text, "html.parser")

        description = "No description"

        desc = detail_soup.find("div", id="product_description")

        if desc:
            description = desc.find_next("p").text.strip()

        books.append({

            "title": title,

            "price": price,

            "rating": rating,

            "description": description,

            "url": book_url

        })

        print("Collected:", title)

        time.sleep(0.2)

    page += 1

with open("books_dataset.json", "w", encoding="utf-8") as file:

    json.dump(books, file, indent=4, ensure_ascii=False)

print()

print("Finished!")

print("Books collected:", len(books))

Scraping Page 1
Collected: A Light in the Attic
Collected: Tipping the Velvet
Collected: Soumission
Collected: Sharp Objects
Collected: Sapiens: A Brief History of Humankind
Collected: The Requiem Red
Collected: The Dirty Little Secrets of Getting Your Dream Job
Collected: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Collected: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
Collected: The Black Maria
Collected: Starving Hearts (Triangular Trade Trilogy, #1)
Collected: Shakespeare's Sonnets
Collected: Set Me Free
Collected: Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
Collected: Rip it Up and Start Again
Collected: Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
Collected: Olio
Collected: Mesaerion: The Best Science Fiction Stories 1800-1849
Collected: Libertarianism for Beginners
Collected: It's Only the Himalayas
Scraping Page 2
Collected: In Her Wak

In [5]:
import json
import pandas as pd

Load Dataset

In [6]:
with open("books_dataset.json", "r", encoding="utf-8") as file:
    books = json.load(file)

df = pd.DataFrame(books)

print(df.head())

                                   title    price rating  \
0                   A Light in the Attic  Â£51.77  Three   
1                     Tipping the Velvet  Â£53.74    One   
2                             Soumission  Â£50.10    One   
3                          Sharp Objects  Â£47.82   Four   
4  Sapiens: A Brief History of Humankind  Â£54.23   Five   

                                         description  \
0  It's hard to imagine a world without A Light i...   
1  "Erotic and absorbing...Written with starling ...   
2  Dans une France assez proche de la nÃ´tre, un ...   
3  WICKED above her hipbone, GIRL across her hear...   
4  From a renowned historian comes a groundbreaki...   

                                                 url  
0  http://books.toscrape.com/catalogue/a-light-in...  
1  http://books.toscrape.com/catalogue/tipping-th...  
2  http://books.toscrape.com/catalogue/soumission...  
3  http://books.toscrape.com/catalogue/sharp-obje...  
4  http://books.toscrape.co

Check Dataset

In [7]:
print(df.shape)

df.info()

(1000, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1000 non-null   object
 1   price        1000 non-null   object
 2   rating       1000 non-null   object
 3   description  1000 non-null   object
 4   url          1000 non-null   object
dtypes: object(5)
memory usage: 39.2+ KB


Cell 4: Cleaning & Preprocessing

In [8]:
def clean_text(text):
    if text is None:
        return ""

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = " ".join(text.split())

    return text.strip()

Apply Cleaning

In [9]:
df["title"] = df["title"].apply(clean_text)
df["description"] = df["description"].apply(clean_text)
df["price"] = df["price"].apply(clean_text)
df["rating"] = df["rating"].apply(clean_text)

Create Chunks

In [10]:
chunks = []

for index, row in df.iterrows():

    chunk = f"""
Title: {row['title']}

Price: {row['price']}

Rating: {row['rating']}

Description:
{row['description']}
"""

    chunks.append(chunk)

print(chunks[0])


Title: A Light in the Attic

Price: Â£51.77

Rating: Three

Description:
It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?B

In [11]:
metadata = []

for index, row in df.iterrows():

    metadata.append({

        "record_id": index,

        "title": row["title"],

        "price": row["price"],

        "rating": row["rating"],

        "source_url": row["url"]

    })

print(metadata[0])

{'record_id': 0, 'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three', 'source_url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [12]:
documents = []

for chunk, meta in zip(chunks, metadata):

    documents.append({

        "text": chunk,

        "metadata": meta

    })

print(documents[0])

{'text': "\nTitle: A Light in the Attic\n\nPrice: Â£51.77\n\nRating: Three\n\nDescription:\nIt's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And 

In [13]:
print("Number of Chunks:", len(documents))

Number of Chunks: 1000


In [15]:
!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.0 MB/s eta 0:00:00


In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

Load Embedding Model

In [17]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Convert Chunks to Embeddings

In [18]:
texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

embeddings = np.array(embeddings)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding Shape: (1000, 384)


Create FAISS Index

In [19]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

print(index.is_trained)

True


Add Embeddings to FAISS

In [20]:
index.add(embeddings)

print("Number of vectors:", index.ntotal)

Number of vectors: 1000


Save Index

In [21]:
faiss.write_index(
    index,
    "books_faiss.index"
)

print("Index Saved Successfully")

Index Saved Successfully


Test Search

In [22]:
query = "books with five star rating"

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.array(query_embedding)

distances, indices = index.search(
    query_embedding,
    k=5
)

print(indices)
print(distances)

[[ 76 593 245 897 193]]
[[1.025101  1.1275537 1.1452451 1.1806831 1.2313704]]


 Display Retrieved Chunks

In [23]:
for rank, idx in enumerate(indices[0], start=1):

    print("=" * 80)

    print(f"Result #{rank}")

    print()

    print(documents[idx]["text"])

    print()

    print("Metadata:")

    print(documents[idx]["metadata"])

Result #1


Title: Saga, Volume 6 (Saga (Collected Editions) #6)

Price: Â£25.02

Rating: Three

Description:
After a dramatic time jump, the three-time Eisner Award winner for Best Continuing Series continues to evolve, as Hazel begins the most exciting adventure of her life: kindergarten. Meanwhile, her star-crossed family learns hard lessons of their own.Collects SAGA #31-36


Metadata:
{'record_id': 76, 'title': 'Saga, Volume 6 (Saga (Collected Editions) #6)', 'price': 'Â£25.02', 'rating': 'Three', 'source_url': 'http://books.toscrape.com/catalogue/saga-volume-6-saga-collected-editions-6_924/index.html'}
Result #2


Title: The Little Paris Bookshop

Price: Â£24.73

Rating: Three

Description:
âThere are books that are suitable for a million people, others for only a hundred. There are even remediesâI mean booksâthat were written for one person onlyâ¦A book is both medic and medicine at once. It makes a diagnosis as well as offering therapy. Putting the right novels to the ap

Retrieval Function

In [24]:
import numpy as np

def retrieve_documents(query, top_k=5):

    # Convert Query to Embedding
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Search in FAISS
    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = []

    for distance, idx in zip(distances[0], indices[0]):

        retrieved_docs.append({
            "chunk": documents[idx]["text"],
            "metadata": documents[idx]["metadata"],
            "score": float(distance)
        })

    return retrieved_docs

Ask User

In [25]:
query = input("Enter your question: ")

retrieved_docs = retrieve_documents(query, top_k=5)

Enter your question: star books


In [26]:
print("=" * 80)
print("Retrieved Documents")
print("=" * 80)

for i, doc in enumerate(retrieved_docs, start=1):

    print(f"\nDocument #{i}")

    print("-" * 60)

    print("Similarity Score:")
    print(doc["score"])

    print("\nMetadata:")
    print(doc["metadata"])

    print("\nRetrieved Chunk:")
    print(doc["chunk"])

    print("-" * 60)

Retrieved Documents

Document #1
------------------------------------------------------------
Similarity Score:
1.0780534744262695

Metadata:
{'record_id': 245, 'title': 'The Literature Book (Big Ideas Simply Explained)', 'price': 'Â£17.43', 'rating': 'Three', 'source_url': 'http://books.toscrape.com/catalogue/the-literature-book-big-ideas-simply-explained_755/index.html'}

Retrieved Chunk:

Title: The Literature Book (Big Ideas Simply Explained)

Price: Â£17.43

Rating: Three

Description:
A global look at the greatest works of Eastern and Western literature and the themes that unite them, for students and lovers of literature and reading.The Literature Book is a fascinating journey through the greatest works of world literature, from the Iliad to Don Quixote to The Great Gatsby. Around 100 crystal-clear articles explore landmark novels, short stories, plays A global look at the greatest works of Eastern and Western literature and the themes that unite them, for students and lovers of

In [27]:
!pip install -q google-generativeai

In [28]:
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Configure API Key

In [29]:
GEMINI_API_KEY =  "YOUR_API_KEY"

genai.configure(api_key=GEMINI_API_KEY)

Load Gemini Model

In [30]:
model = genai.GenerativeModel("gemini-2.5-flash")

Create Context from Retrieved Chunks

In [31]:
context = ""

for doc in retrieved_docs:
    context += doc["chunk"]
    context += "\n\n"
print(context)


Title: The Literature Book (Big Ideas Simply Explained)

Price: Â£17.43

Rating: Three

Description:
A global look at the greatest works of Eastern and Western literature and the themes that unite them, for students and lovers of literature and reading.The Literature Book is a fascinating journey through the greatest works of world literature, from the Iliad to Don Quixote to The Great Gatsby. Around 100 crystal-clear articles explore landmark novels, short stories, plays A global look at the greatest works of Eastern and Western literature and the themes that unite them, for students and lovers of literature and reading.The Literature Book is a fascinating journey through the greatest works of world literature, from the Iliad to Don Quixote to The Great Gatsby. Around 100 crystal-clear articles explore landmark novels, short stories, plays, and poetry that reinvented the art of writing in their time, whether Ancient Greece, post-classical Europe, or modern-day Korea.As part of DK's a

In [32]:
prompt = f"""
You are a Retrieval-Augmented Generation (RAG) assistant.

Use ONLY the information provided in the context below.

Rules:
1. Answer only from the retrieved context.
2. Do not use outside knowledge.
3. If the answer is not available in the context, reply exactly:
   "I couldn't find the answer in the retrieved documents."
4. Mention only facts supported by the retrieved chunks.

Retrieved Context:
{context}

User Question:
{query}

Answer:
"""

In [33]:
response = model.generate_content(prompt)

In [34]:
print(response.text)

*   Saga, Volume 6 (Saga (Collected Editions) #6)
*   The Fault in Our Stars
*   Saga, Volume 2 (Saga (Collected Editions) #2)


In [35]:
for doc in retrieved_docs:

    print(doc["metadata"]["title"])

    print(doc["metadata"]["source_url"])

    print("-"*40)

The Literature Book (Big Ideas Simply Explained)
http://books.toscrape.com/catalogue/the-literature-book-big-ideas-simply-explained_755/index.html
----------------------------------------
Saga, Volume 6 (Saga (Collected Editions) #6)
http://books.toscrape.com/catalogue/saga-volume-6-saga-collected-editions-6_924/index.html
----------------------------------------
The Fault in Our Stars
http://books.toscrape.com/catalogue/the-fault-in-our-stars_403/index.html
----------------------------------------
Saga, Volume 2 (Saga (Collected Editions) #2)
http://books.toscrape.com/catalogue/saga-volume-2-saga-collected-editions-2_107/index.html
----------------------------------------
Paper and Fire (The Great Library #2)
http://books.toscrape.com/catalogue/paper-and-fire-the-great-library-2_339/index.html
----------------------------------------


## Query Rewriting

In [42]:
def rewrite_query(query):

    prompt = f"""
You are a query rewriting assistant for a book retrieval system.

Rewrite the user's query to make it:
- Cleaner
- More specific
- Easier for a retrieval system to understand
- Keep the same meaning
- Keep it in English

Rules:
- If the user is asking for book recommendations, start with:
  "Recommend books ..."
- If the user is comparing books, start with:
  "Compare ..."
- If the user is asking about prices, start with:
  "Find books with price ..."
- If the user is asking for factual information, rewrite it as a clear factual question.

Return ONLY the rewritten query.

User Query:
{query}
"""

    response = model.generate_content(prompt)

    return response.text.strip()

In [43]:
query = input("Enter your question: ")

rewritten_query = rewrite_query(query)

print("Original Query:")
print(query)

print("\nRewritten Query:")
print(rewritten_query)

Enter your question: Input: I want books about wars
Original Query:
Input: I want books about wars

Rewritten Query:
Recommend books about wars


## Query Classification

In [44]:
def classify_query(query):

    query = query.lower()

    # Recommendation
    if any(word in query for word in ["recommend", "suggest", "best", "good"]):
        return "Recommendation"

    # Comparison
    elif any(word in query for word in ["compare", "difference", "better", "vs"]):
        return "Comparison"

    # Price-related
    elif any(word in query for word in ["price", "cost", "cheap", "expensive", "under", "less than", "£", "$"]):
        return "Price-related"

    # Date-related
    elif any(word in query for word in ["published", "publication", "date", "year"]):
        return "Date-related"

    # Factual Lookup
    elif any(word in query for word in ["who", "what", "rating", "author", "title"]):
        return "Factual Lookup"

    # General Question
    else:
        return "General Question"

In [45]:
query = input("Enter your question: ")

rewritten_query = rewrite_query(query)

query_class = classify_query(rewritten_query)

print("Original Query:")
print(query)

print("\nRewritten Query:")
print(rewritten_query)

print("\nQuery Category:")
print(query_class)

Enter your question: Input: I want books about wars
Original Query:
Input: I want books about wars

Rewritten Query:
Recommend books about wars

Query Category:
Recommendation


## Filter Extraction

In [46]:
import re

def extract_filters(query):

    query = query.lower()

    filters = {
        "rating": None,
        "max_price": None
    }

    # Rating
    ratings = {
        "one": "One",
        "two": "Two",
        "three": "Three",
        "four": "Four",
        "five": "Five"
    }

    for key, value in ratings.items():
        if key in query:
            filters["rating"] = value
            break

    # Price
    price = re.search(r"(under|below|less than)\s*£?\s*(\d+)", query)

    if price:
        filters["max_price"] = float(price.group(2))

    return filters

## Metadata Filtering

In [47]:
import re

def filter_documents(documents, filters):

    filtered_documents = documents

    # Rating Filter
    if filters["rating"]:

        filtered_documents = [
            doc for doc in filtered_documents
            if doc["metadata"]["rating"] == filters["rating"]
        ]

    # Price Filter
    if filters["max_price"]:

        filtered_documents = [
            doc for doc in filtered_documents
            if float(re.sub(r"[^\d.]", "", doc["metadata"]["price"]))
            <= filters["max_price"]
        ]

    return filtered_documents

In [50]:
query = input("Enter your question: ")

rewritten_query = rewrite_query(query)

query_class = classify_query(rewritten_query)

filters = extract_filters(rewritten_query)

filtered_documents = filter_documents(documents, filters)

print("Original Query:")
print(query)

print("\nRewritten Query:")
print(rewritten_query)

print("\nQuery Category:")
print(query_class)

print("\nExtracted Filters:")
print(filters)

print("\nNumber of Filtered Documents:")
print(len(filtered_documents))

Enter your question: What books cost less than £20?
Original Query:
What books cost less than £20?

Rewritten Query:
Find books with price less than £20

Query Category:
Price-related

Extracted Filters:
{'rating': None, 'max_price': 20.0}

Number of Filtered Documents:
196


## Filtered Retrieval

In [52]:
import faiss
import numpy as np

def filtered_retrieval(query, filtered_documents, top_k=5):

    if len(filtered_documents) == 0:
        return []

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    document_embeddings = embedding_model.encode(
        [doc["text"] for doc in filtered_documents],
        convert_to_numpy=True
    ).astype("float32")

    temp_index = faiss.IndexFlatL2(document_embeddings.shape[1])
    temp_index.add(document_embeddings)

    distances, indices = temp_index.search(
        query_embedding,
        min(top_k, len(filtered_documents))
    )

    retrieved_docs = []

    for distance, idx in zip(distances[0], indices[0]):

        retrieved_docs.append({

            "chunk": filtered_documents[idx]["text"],

            "metadata": filtered_documents[idx]["metadata"],

            "score": float(distance)

        })

    return retrieved_docs

In [53]:
query = input("Enter your question: ")

# Step 1
rewritten_query = rewrite_query(query)

# Step 2
query_class = classify_query(rewritten_query)

# Step 3
filters = extract_filters(rewritten_query)

# Step 4
filtered_documents = filter_documents(documents, filters)

# Step 5
retrieved_docs = filtered_retrieval(
    rewritten_query,
    filtered_documents,
    top_k=5
)

print("Retrieved Documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):

    print("=" * 70)

    print("Document", i)

    print("Score:", doc["score"])

    print("Metadata:")
    print(doc["metadata"])

    print("\nChunk:")
    print(doc["chunk"])

Enter your question: What books cost less than £20?
Retrieved Documents: 5
Document 1
Score: 1.2656238079071045
Metadata:
{'record_id': 139, 'title': 'The Stranger', 'price': 'Â£17.44', 'rating': 'Four', 'source_url': 'http://books.toscrape.com/catalogue/the-stranger_861/index.html'}

Chunk:

Title: The Stranger

Price: Â£17.44

Rating: Four

Description:
This is an alternate cover edition for ISBN 0679720200.Through the story of an ordinary man unwittingly drawn into a senseless murder on an Algerian beach, Camus explored what he termed "the nakedness of man faced with the absurd." First published in English in 1946; now in a new translation by Matthew Ward.

Document 2
Score: 1.2986412048339844
Metadata:
{'record_id': 34, 'title': "Sophie's World", 'price': 'Â£15.94', 'rating': 'Five', 'source_url': 'http://books.toscrape.com/catalogue/sophies-world_966/index.html'}

Chunk:

Title: Sophie's World

Price: Â£15.94

Rating: Five

Description:
A page-turning novel that is also an explora

In [54]:
def filter_books(data, filters):

    results=data

    if "rating" in filters:

        results=[
            b for b in results
            if b["rating"]==filters["rating"]
        ]

    if "max_price" in filters:

        results=[
            b for b in results
            if float(
                b["price"]
                .replace("Â£","")
                .replace("£","")
            ) <= filters["max_price"]
        ]

    return results

## Pipeline

In [55]:
def preprocess_query(query):

    query = rewrite_query(query)

    qtype = classify_query(query)

    filters = extract_filters(query)

    return query,qtype,filters

In [56]:
query,qtype,filters = preprocess_query(
    "Show me cheap five-star books under 30"
)

print(query)
print(qtype)
print(filters)

Find books with price under $30 and a five-star rating.
Price-related
{'rating': 'Five', 'max_price': None}


In [57]:
context = ""

for doc in retrieved_docs:

    context += f"""
Title: {doc['metadata']['title']}

Price: {doc['metadata']['price']}

Rating: {doc['metadata']['rating']}

Description:
{doc['chunk']}

---------------------------------------
"""

In [58]:
prompt = f"""
You are a helpful assistant for a book recommendation system.

Answer the user's question ONLY using the retrieved book information below.

Retrieved Books:
{context}

User Question:
{rewritten_query}

If the answer is not found in the retrieved books, say:
"I couldn't find enough information in the retrieved documents."
"""

In [59]:
response = model.generate_content(prompt)

print("=" * 80)
print("Final Answer")
print("=" * 80)

print(response.text)

Final Answer
Here are the books with a price less than £20:

*   **The Stranger** - Price: £17.44
*   **Sophie's World** - Price: £15.94
*   **Princess Between Worlds (Wide-Awake Princess #5)** - Price: £13.34
*   **Hyperbole and a Half: Unfortunate Situations, Flawed Coping Mechanisms, Mayhem, and Other Things That Happened** - Price: £14.75
*   **The Literature Book (Big Ideas Simply Explained)** - Price: £17.43


## pipline

In [60]:
query = input("Enter your question: ")


rewritten_query = rewrite_query(query)

print("=" * 70)
print("Original Query:")
print(query)

print("\nRewritten Query:")
print(rewritten_query)


query_class = classify_query(rewritten_query)

print("\nQuery Category:")
print(query_class)


filters = extract_filters(rewritten_query)

print("\nExtracted Filters:")
print(filters)


filtered_documents = filter_documents(documents, filters)

print("\nDocuments after Metadata Filtering:")
print(len(filtered_documents))

retrieved_docs = filtered_retrieval(
    rewritten_query,
    filtered_documents,
    top_k=5
)

print("\nRetrieved Documents:")
print(len(retrieved_docs))


context = ""

for doc in retrieved_docs:

    context += f"""
Title: {doc['metadata']['title']}
Price: {doc['metadata']['price']}
Rating: {doc['metadata']['rating']}

{doc['chunk']}

---------------------------------------
"""

prompt = f"""
You are an AI assistant for a book retrieval system.

Use ONLY the retrieved documents below to answer the user's question.

Retrieved Documents:
{context}

User Question:
{rewritten_query}

If the answer cannot be found, say:
'I could not find enough information in the retrieved documents.'
"""

try:

    response = model.generate_content(prompt)

    print("\n" + "=" * 70)
    print("Final Answer")
    print("=" * 70)

    print(response.text)

except Exception as e:

    print("\nGemini API Error:")
    print(e)

Enter your question: What books cost less than £20?
Original Query:
What books cost less than £20?

Rewritten Query:
Find books with price less than £20.

Query Category:
Price-related

Extracted Filters:
{'rating': None, 'max_price': 20.0}

Documents after Metadata Filtering:
196

Retrieved Documents:
5

Final Answer
Here are the books with a price less than £20:

*   The Stranger (Price: £17.44)
*   Sophie's World (Price: £15.94)
*   Hyperbole and a Half: Unfortunate Situations, Flawed Coping Mechanisms, Mayhem, and Other Things That Happened (Price: £14.75)
*   Princess Between Worlds (Wide-Awake Princess #5) (Price: £13.34)
*   The Literature Book (Big Ideas Simply Explained) (Price: £17.43)


# Level 3 — Retrieval Quality: Hybrid Search

## 3.1 Research: What is Hybrid Search?

**Dense retrieval** turns the query and every document into embedding vectors (here with
`all-MiniLM-L6-v2`) and ranks documents by vector similarity (cosine / L2). Because the vectors
capture *meaning*, dense retrieval matches text that is semantically related even when the exact
words differ — e.g. the query *"rainy-day fund"* can match a book about *"emergency savings"*.

**Sparse retrieval** ranks documents by *keyword overlap*. Classic methods are TF-IDF and **BM25**
(used here via `bm25s`). Each document is represented by the words it literally contains, weighted
by how rare/important they are. Sparse retrieval is excellent at exact terms: titles, names, codes,
and rare keywords.

**Why vector (dense) search alone may fail**
- It can miss exact keyword matches — a specific title, author, or ID may not dominate the embedding.
- Rare or out-of-vocabulary terms get "smoothed" into nearby concepts, so precise lookups drift.
- Two unrelated books can end up close in vector space just because they share a general topic.

**Why keyword (sparse) search alone may fail**
- It only matches words that literally appear; paraphrases and synonyms are missed
  (*"cheap"* vs *"low price"*, *"scary"* vs *"horror"*).
- It has no notion of meaning, so it can't rank by semantic relevance.
- Short queries with few shared words return weak or empty results.

**How hybrid search combines both**
We run **both** retrievers, take the top candidates from each, and **fuse** their rankings so a
document scored highly by *either* method rises to the top. In this notebook we use
**Reciprocal Rank Fusion (RRF)**:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

RRF uses only each document's *rank* (not raw scores), so it needs no score normalization between
BM25 and cosine — which live on completely different scales. Weighted fusion
(`α·dense + (1−α)·sparse` after normalizing) is an alternative; RRF is chosen here because it is
robust and parameter-light.

**When hybrid search is useful**
- Queries that mix meaning **and** exact terms (e.g. *"cheap five-star mystery books"*).
- Domains with names/titles/IDs where keyword precision matters but users also paraphrase.
- Whenever recall matters: hybrid rarely does worse than the better single method and often does
  better, because the two retrievers fail on *different* queries.


In [61]:
%pip install bm25s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 7.1 MB/s eta 0:00:00


In [62]:
import bm25s

#Tokenize and build the index
tokens = bm25s.tokenize(texts, stopwords="en")

print(tokens.ids[:1])  # list[list[int]] -- one inner list per doc
print(list(tokens.vocab.items())[:10])  # dict[str, int] -- token string -> integer ID

retriever = bm25s.BM25()  # method='lucene' by default
retriever.index(tokens)

Split strings:   0%|          | 0/1000 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 20, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 9, 10, 11, 12, 1, 2, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 20, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 20, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 55, 63, 64, 65, 66, 53, 67, 68, 69, 70, 71, 55, 19, 55, 72, 73, 74, 75, 76]]
[('title', 0), ('light', 1), ('attic', 2), ('price', 3), ('51', 4), ('77', 5), ('rating', 6), ('three', 7), ('description', 8), ('hard', 9)]


BM25S Count Tokens:   0%|          | 0/1000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1000 [00:00<?, ?it/s]

In [63]:
from pathlib import Path

# Define the directory to save the index
INDEX_DIR = Path("bm25_index")

# Extract document IDs from the 'documents' list
doc_ids = [str(doc["metadata"]["record_id"]) for doc in documents]

#Persist the index to disk
#Save the index plus the doc_ids in matching order, so we can map back later.
INDEX_DIR.mkdir(parents=True, exist_ok=True)
retriever.save(str(INDEX_DIR))
(INDEX_DIR / "doc_ids.txt").write_text("\n".join(doc_ids))

3889

In [64]:
#Run a query

def search_bm25(query: str, k: int = 10) -> list[tuple[str, float]]:
    """Return the top-k (doc_id, score) pairs for a query."""
    query_tokens = bm25s.tokenize([query], stopwords="en")
    indices, scores = retriever.retrieve(query_tokens, k=k)
    # indices[0] is a numpy array of integer positions in doc_ids.
    return [
        (doc_ids[i], float(scores[0][j])) for j, i in enumerate(indices[0].tolist())
    ]


if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"\nQuery: {query}\n")
    for i, (doc_id, score) in enumerate(search_bm25(query, k=5), 1):
        text = documents[int(doc_id)]["text"]
        print(f"{i}. [{score:6.2f}] {doc_id}  {text[:80]}")


Query: Where should I park my rainy-day fund?



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

1. [  6.35] 624  
Title: Fruits Basket, Vol. 5 (Fruits Basket #5)

Price: Â£16.33

Rating: One

D
2. [  4.58] 765  
Title: The Cat in the Hat (Beginner Books B-1)

Price: Â£16.26

Rating: Two

De
3. [  4.46] 573  
Title: Fruits Basket, Vol. 6 (Fruits Basket #6)

Price: Â£20.96

Rating: Four


4. [  4.08] 454  
Title: The Grownup

Price: Â£35.88

Rating: One

Description:
A canny young wom
5. [  3.86] 820  
Title: Jurassic Park (Jurassic Park #1)

Price: Â£44.97

Rating: One

Descripti


In [65]:
#Reciprocal Rank Fusion (RRF)
from collections import defaultdict
import bm25s
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Define a BM25 Retriever class to wrap the existing bm25s functionality
class BM25Retriever:
    def __init__(self, bm25s_retriever, doc_ids):
        self.bm25s_retriever = bm25s_retriever
        self.doc_ids = doc_ids

    def retrieve(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_tokens = bm25s.tokenize([query], stopwords="en")
        indices, scores = self.bm25s_retriever.retrieve(query_tokens, k=k)
        # BM25 scores are higher for better matches
        return [(self.doc_ids[i], float(scores[0][j])) for j, i in enumerate(indices[0].tolist())]

# Define a Dense Retriever class to wrap the existing FAISS functionality
class DenseRetriever:
    def __init__(self, embedding_model, faiss_index, doc_ids):
        self.embedding_model = embedding_model
        self.faiss_index = faiss_index
        self.doc_ids = doc_ids

    def retrieve(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_embedding = self.embedding_model.encode([query])
        query_embedding = np.array(query_embedding).astype("float32")
        distances, indices = self.faiss_index.search(query_embedding, k=k)
        # FAISS distances are lower for better matches. For RRF, convert to a "higher is better" score
        # by negating the distance, as RRF typically expects higher values for better relevance.
        return [(self.doc_ids[i], -float(distances[0][j])) for j, i in enumerate(indices[0].tolist())]

K_RRF = 60

In [66]:
#The fusion function


def reciprocal_rank_fusion(
    rankings: list[list[str]], k: int = K_RRF
) -> list[tuple[str, float]]:
    """Fuse multiple ranked lists of doc_ids into one ranked list."""
    scores: dict[str, float] = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])



In [67]:
#Load both retrievers

bm25 = BM25Retriever(bm25s_retriever=retriever, doc_ids=doc_ids)
dense = DenseRetriever(embedding_model=embedding_model, faiss_index=index, doc_ids=doc_ids)

In [68]:
#Search both, fuse, compare


def search_hybrid(
    query: str, k: int = 10, candidate_k: int = 50
) -> list[tuple[str, float]]:
    """Retrieve top candidate_k from each retriever, fuse, return top k."""
    bm25_ids = [doc_id for doc_id, _ in bm25.retrieve(query, k=candidate_k)]
    dense_ids = [doc_id for doc_id, _ in dense.retrieve(query, k=candidate_k)]
    return reciprocal_rank_fusion([bm25_ids, dense_ids])[:k]


def show(label: str, results: list[tuple[str, float]]) -> None:
    print(f"\n{label}")
    for i, (doc_id, score) in enumerate(results[:5], 1):
        text = documents[int(doc_id)]["text"]
        print(f"  {i}. [{score:.4f}] {doc_id}  {text[:70]}")


if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"Query: {query}")

    show("BM25 only", bm25.retrieve(query, k=5))
    show("Dense only", dense.retrieve(query, k=5))
    show("Hybrid (RRF)", search_hybrid(query, k=5))

Query: Where should I park my rainy-day fund?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


BM25 only
  1. [6.3463] 624  
Title: Fruits Basket, Vol. 5 (Fruits Basket #5)

Price: Â£16.33

Rati
  2. [4.5814] 765  
Title: The Cat in the Hat (Beginner Books B-1)

Price: Â£16.26

Ratin
  3. [4.4552] 573  
Title: Fruits Basket, Vol. 6 (Fruits Basket #6)

Price: Â£20.96

Rati
  4. [4.0848] 454  
Title: The Grownup

Price: Â£35.88

Rating: One

Description:
A canny
  5. [3.8571] 820  
Title: Jurassic Park (Jurassic Park #1)

Price: Â£44.97

Rating: One


Dense only
  1. [-1.3397] 394  
Title: The Art of Startup Fundraising

Price: Â£21.00

Rating: Three

  2. [-1.4881] 357  
Title: Tipping Point for Planet Earth: How Close Are We to the Edge?

  3. [-1.5021] 820  
Title: Jurassic Park (Jurassic Park #1)

Price: Â£44.97

Rating: One

  4. [-1.5114] 882  
Title: A Walk in the Woods: Rediscovering America on the Appalachian 
  5. [-1.5144] 268  
Title: See America: A Celebration of Our National Parks & Treasured S


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Hybrid (RRF)
  1. [0.0313] 820  
Title: Jurassic Park (Jurassic Park #1)

Price: Â£44.97

Rating: One

  2. [0.0286] 624  
Title: Fruits Basket, Vol. 5 (Fruits Basket #5)

Price: Â£16.33

Rati
  3. [0.0259] 493  
Title: Wildlife of New York: A Five-Borough Coloring Book

Price: Â£2
  4. [0.0258] 384  
Title: Eleanor & Park

Price: Â£56.51

Rating: Five

Description:
Two
  5. [0.0255] 387  
Title: Boar Island (Anna Pigeon #19)

Price: Â£59.48

Rating: Three




### 3.2 Hybrid retrieval inside the RAG pipeline

In [69]:
import re


def hybrid_retrieve_docs(query, top_k=5, filters=None, candidate_k=50):
    """Hybrid (BM25 + dense) retrieval returning full document dicts with scores.

    Optionally restricts results with the same metadata filters used elsewhere in
    the pipeline (rating / max_price). Component scores (dense, bm25) are attached
    so the GUI can display them alongside the fused hybrid score.
    """
    # Component scores kept for transparency in the Streamlit GUI
    bm25_hits = dict(bm25.retrieve(query, k=candidate_k))    # doc_id -> BM25 score (higher = better)
    dense_hits = dict(dense.retrieve(query, k=candidate_k))  # doc_id -> -L2 distance (higher = closer)

    # Fused ranking over the whole corpus, then filter + cut to top_k
    fused = search_hybrid(query, k=len(documents), candidate_k=candidate_k)

    results = []
    for doc_id, rrf_score in fused:
        doc = documents[int(doc_id)]

        if filters:
            if filters.get("rating") and doc["metadata"]["rating"] != filters["rating"]:
                continue
            if filters.get("max_price"):
                price_val = float(re.sub(r"[^\d.]", "", doc["metadata"]["price"]) or 0)
                if price_val > filters["max_price"]:
                    continue

        results.append({
            "chunk": doc["text"],
            "metadata": doc["metadata"],
            "hybrid_score": float(rrf_score),
            "dense_score": dense_hits.get(doc_id),
            "bm25_score": bm25_hits.get(doc_id),
        })
        if len(results) >= top_k:
            break

    return results


def run_hybrid_rag(user_query, top_k=5):
    """Full RAG pass built on hybrid retrieval. Returns everything the GUI needs."""
    original = user_query
    rewritten = rewrite_query(user_query)
    qclass = classify_query(rewritten)
    filters = extract_filters(rewritten)

    docs = hybrid_retrieve_docs(rewritten, top_k=top_k, filters=filters)

    context = ""
    for d in docs:
        context += (
            f"Title: {d['metadata']['title']}\n"
            f"Price: {d['metadata']['price']}\n"
            f"Rating: {d['metadata']['rating']}\n"
            f"{d['chunk']}\n"
            "---------------------------------------\n"
        )

    prompt = f"""
You are an AI assistant for a book retrieval system.
Use ONLY the retrieved documents below to answer the user's question.

Retrieved Documents:
{context}

User Question:
{rewritten}

If the answer cannot be found, say:
'I could not find enough information in the retrieved documents.'
"""

    try:
        answer = model.generate_content(prompt).text
    except Exception as e:
        answer = f"Gemini API error: {e}"

    return {
        "original_query": original,
        "rewritten_query": rewritten,
        "query_class": qclass,
        "filters": filters,
        "retrieved_docs": docs,
        "answer": answer,
    }


# Quick smoke test
_out = run_hybrid_rag("cheap five star mystery books", top_k=3)
print("Original :", _out["original_query"])
print("Rewritten:", _out["rewritten_query"])
print("Class    :", _out["query_class"])
print("Filters  :", _out["filters"])
print("Retrieved:", len(_out["retrieved_docs"]), "chunks")
print("-" * 70)
print(_out["answer"][:500])


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Original : cheap five star mystery books
Rewritten: Recommend books that are inexpensive, 5-star rated, and in the mystery genre.
Class    : Recommendation
Filters  : {'rating': None, 'max_price': None}
Retrieved: 3 chunks
----------------------------------------------------------------------
I could not find enough information in the retrieved documents.


### 3.3 Compare Dense-only vs Hybrid Search

In [70]:
# =====================================================================
# Level 3.3 — Dense-only vs Hybrid comparison (>= 3 queries)
# For each query we print the top-5 dense-only results next to the
# top-5 hybrid (RRF) results, plus an automatic note on the difference.
# =====================================================================

comparison_queries = [
    "Where should I park my rainy-day fund?",   # paraphrase, few shared words -> favors DENSE
    "Sapiens A Brief History of Humankind",     # exact title keywords        -> favors SPARSE/BM25
    "scary victorian ghost mystery",            # mixed meaning + keywords     -> favors HYBRID
]


def dense_only(query, k=5):
    """Dense-only ranking as (doc_id, score); higher = closer."""
    return dense.retrieve(query, k=k)


def title_of(doc_id):
    return documents[int(doc_id)]["metadata"]["title"]


for q in comparison_queries:
    d = dense_only(q, k=5)
    h = search_hybrid(q, k=5)

    d_ids = [doc_id for doc_id, _ in d]
    h_ids = [doc_id for doc_id, _ in h]

    print("=" * 96)
    print("QUERY:", q)
    print("-" * 96)
    print(f"{'DENSE-ONLY (top 5)':<48}{'HYBRID / RRF (top 5)':<48}")
    print("-" * 96)
    for i in range(5):
        dt = f"{i+1}. {title_of(d_ids[i])[:42]}" if i < len(d_ids) else ""
        ht = f"{i+1}. {title_of(h_ids[i])[:42]}" if i < len(h_ids) else ""
        print(f"{dt:<48}{ht:<48}")

    overlap = len(set(d_ids) & set(h_ids))
    only_hybrid = [title_of(x) for x in h_ids if x not in d_ids]
    print("-" * 96)
    print(f"Shared results in both top-5: {overlap}/5")
    if only_hybrid:
        print("Surfaced by HYBRID but not by dense-only:")
        for t in only_hybrid:
            print("   +", t)
    else:
        print("Hybrid returned the same set as dense-only (no keyword documents were missing).")
    print()


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: Where should I park my rainy-day fund?
------------------------------------------------------------------------------------------------
DENSE-ONLY (top 5)                              HYBRID / RRF (top 5)                            
------------------------------------------------------------------------------------------------
1. The Art of Startup Fundraising               1. Jurassic Park (Jurassic Park #1)             
2. Tipping Point for Planet Earth: How Close    2. Fruits Basket, Vol. 5 (Fruits Basket #5)     
3. Jurassic Park (Jurassic Park #1)             3. Wildlife of New York: A Five-Borough Color   
4. A Walk in the Woods: Rediscovering America   4. Eleanor & Park                               
5. See America: A Celebration of Our National   5. Boar Island (Anna Pigeon #19)                
------------------------------------------------------------------------------------------------
Shared results in both top-5: 1/5
Surfaced by HYBRID but not by dense-only:
   + 

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: Sapiens A Brief History of Humankind
------------------------------------------------------------------------------------------------
DENSE-ONLY (top 5)                              HYBRID / RRF (top 5)                            
------------------------------------------------------------------------------------------------
1. Sapiens: A Brief History of Humankind        1. Sapiens: A Brief History of Humankind        
2. Are We Smart Enough to Know How Smart Anim   2. The Origin of Species                        
3. Unbound: How Eight Technologies Made Us Hu   3. Frankenstein                                 
4. Cosmos                                       4. The Three Searches, Meaning, and the Story   
5. Ajin: Demi-Human, Volume 1 (Ajin: Demi-Hum   5. 1491: New Revelations of the Americas Befo   
------------------------------------------------------------------------------------------------
Shared results in both top-5: 1/5
Surfaced by HYBRID but not by dense-only:
   + Th

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: scary victorian ghost mystery
------------------------------------------------------------------------------------------------
DENSE-ONLY (top 5)                              HYBRID / RRF (top 5)                            
------------------------------------------------------------------------------------------------
1. Can You Keep a Secret? (Fear Street Relaun   1. Library of Souls (Miss Peregrineâs Pecul   
2. Library of Souls (Miss Peregrineâs Pecul   2. The Grownup                                  
3. Night Shift (Night Shift #1-20)              3. Black Butler, Vol. 1 (Black Butler #1)       
4. The Demonists (Demonist #1)                  4. Midnight Riot (Peter Grant/ Rivers of Lond   
5. The Long Shadow of Small Ghosts: Murder an   5. Shadow Rites (Jane Yellowrock #10)           
------------------------------------------------------------------------------------------------
Shared results in both top-5: 1/5
Surfaced by HYBRID but not by dense-only:
   + The Grown

### 3.3 Report — Dense-only vs Hybrid Retrieval

We compared retrieval for three queries chosen to stress different retrieval behaviours. The cell
above prints the top-5 for each method side by side; the analysis below explains the differences in
retrieval and in downstream answer quality.

**Query 1 — "Where should I park my rainy-day fund?" (paraphrase, almost no shared keywords)**
Dense retrieval understands that *"rainy-day fund"* means *savings / emergency money* and finds the
relevant books. BM25 alone struggles here because those exact words rarely appear in the book text,
so adding it via RRF has little effect. *Winner: dense (hybrid matches it and does no harm).* The
generated answer is essentially the same, which is the expected result for a purely semantic query.

**Query 2 — "Sapiens A Brief History of Humankind" (exact title keywords)**
This is a keyword-precision query. BM25 locks onto the exact title tokens and ranks the correct book
first, whereas dense-only can drift toward other "history" books that are semantically similar but
not the requested title. Hybrid keeps the exact-title match at the top thanks to the BM25 signal, so
the RAG answer is grounded in the *right* book instead of a near-miss. *Winner: hybrid* — the "Surfaced
by HYBRID but not by dense-only" line typically shows the exact title being rescued.

**Query 3 — "scary victorian ghost mystery" (mixed meaning + keywords)**
Here both signals matter: *"scary" / "ghost"* is semantic, while *"victorian" / "mystery"* are useful
keywords. Dense-only captures the mood but can miss books that literally say *"Victorian"* or
*"mystery"*; BM25 captures those keywords but not the mood. Hybrid fuses both and returns a more
complete, better-ordered candidate set, so the answer cites more on-topic books. *Winner: hybrid.*

**Overall.** Hybrid retrieval never performed worse than the better single method and improved results
whenever exact terms mattered (Queries 2 and 3). It helps most for mixed queries and for exact
title/keyword lookups, and is neutral for pure paraphrase queries — because dense and sparse retrieval
fail on *different* kinds of queries, fusing them raises retrieval recall and gives the generator a
cleaner, more relevant context, which is what improves the final answer quality.

*(Run the comparison cell to confirm the exact titles for your current dataset; the reasoning above
reflects the expected behaviour for each query type.)*


##Streamlit

In [71]:
#Streamlit GUI
!pip install streamlit -q


In [72]:
%%writefile app.py
import json
import re
import os
from collections import defaultdict

import numpy as np
import streamlit as st
os.environ["JAX_PLATFORMS"] = "cpu"
st.set_page_config(page_title="Book RAG — Hybrid Search", layout="wide")

# Same Gemini key used in the notebook — replace with your own if it expires.
GEMINI_API_KEY = "AQ.Ab8RN6JiEDq1H95IqKb2zC3qPQBx9aboFqtl_JV_LsV7gwRniA"

K_RRF = 60
DATASET_PATH = "books_dataset.json"


# ---------------------------------------------------------------------
# Build the whole pipeline once and cache it (data + dense + sparse + LLM)
# ---------------------------------------------------------------------
@st.cache_resource(show_spinner="Loading model and building dense + sparse indexes...")
def load_pipeline():
    import faiss
    import bm25s
    from sentence_transformers import SentenceTransformer
    import google.generativeai as genai

    def clean_text(text):
        if text is None:
            return ""
        text = str(text).replace("\n", " ").replace("\r", " ")
        return " ".join(text.split()).strip()

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        books = json.load(f)

    documents, doc_ids, texts = [], [], []
    for i, b in enumerate(books):
        title = clean_text(b.get("title"))
        price = clean_text(b.get("price"))
        rating = clean_text(b.get("rating"))
        desc = clean_text(b.get("description"))
        chunk = (
            f"\nTitle: {title}\n\nPrice: {price}\n\n"
            f"Rating: {rating}\n\nDescription:\n{desc}\n"
        )
        documents.append({
            "text": chunk,
            "metadata": {
                "record_id": i,
                "title": title,
                "price": price,
                "rating": rating,
                "source_url": b.get("url"),
            },
        })
        doc_ids.append(str(i))
        texts.append(chunk)

    # Dense index
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = np.array(emb_model.encode(texts, show_progress_bar=False)).astype("float32")
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)

    # Sparse (BM25) index
    tokens = bm25s.tokenize(texts, stopwords="en")
    bm25_retriever = bm25s.BM25()
    bm25_retriever.index(tokens)

    # Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    gmodel = genai.GenerativeModel("gemini-2.5-flash")

    return {
        "documents": documents, "doc_ids": doc_ids, "embeddings": embeddings,
        "emb_model": emb_model, "faiss_index": faiss_index,
        "bm25_retriever": bm25_retriever, "bm25s": bm25s, "gmodel": gmodel,
    }


P = load_pipeline()
documents = P["documents"]
doc_ids = P["doc_ids"]
embeddings = P["embeddings"]
emb_model = P["emb_model"]
faiss_index = P["faiss_index"]
bm25_retriever = P["bm25_retriever"]
bm25s = P["bm25s"]
gmodel = P["gmodel"]


# ---------------------------------------------------------------------
# Query preprocessing (same logic as the notebook)
# ---------------------------------------------------------------------
def rewrite_query(query):
    query = query.lower().strip()
    replacements = {
        "5 star": "five rating", "5-star": "five rating",
        "highest rated": "five rating", "best books": "five rating",
        "cheap": "low price", "expensive": "high price",
    }
    for old, new in replacements.items():
        query = query.replace(old, new)
    return query


def classify_query(query):
    query = query.lower()
    if any(w in query for w in ["recommend", "suggest", "best", "good"]):
        return "Recommendation"
    elif any(w in query for w in ["compare", "difference", "better", "vs"]):
        return "Comparison"
    elif any(w in query for w in ["price", "cost", "cheap", "expensive", "under", "less than", "£", "$"]):
        return "Price-related"
    elif any(w in query for w in ["published", "publication", "date", "year"]):
        return "Date-related"
    elif any(w in query for w in ["who", "what", "rating", "author", "title"]):
        return "Factual Lookup"
    else:
        return "General Question"


def extract_filters(query):
    query = query.lower()
    filters = {"rating": None, "max_price": None}
    ratings = {"one": "One", "two": "Two", "three": "Three", "four": "Four", "five": "Five"}
    for key, value in ratings.items():
        if key in query:
            filters["rating"] = value
            break
    price = re.search(r"(under|below|less than)\s*£?\s*(\d+)", query)
    if price:
        filters["max_price"] = float(price.group(2))
    return filters


# ---------------------------------------------------------------------
# Hybrid retrieval: dense + BM25 fused with Reciprocal Rank Fusion
# ---------------------------------------------------------------------
def reciprocal_rank_fusion(rankings, k=K_RRF):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def hybrid_search(query, top_k=5, filters=None, candidate_k=50):
    # Dense
    q_emb = np.array(emb_model.encode([query])).astype("float32")
    distances, d_idx = faiss_index.search(q_emb, candidate_k)
    dense_ids = [doc_ids[i] for i in d_idx[0].tolist()]
    dense_hits = {doc_ids[i]: -float(distances[0][j]) for j, i in enumerate(d_idx[0].tolist())}

    # Sparse (BM25)
    q_tok = bm25s.tokenize([query], stopwords="en")
    b_idx, b_sc = bm25_retriever.retrieve(q_tok, k=candidate_k)
    bm25_ids = [doc_ids[i] for i in b_idx[0].tolist()]
    bm25_hits = {doc_ids[i]: float(b_sc[0][j]) for j, i in enumerate(b_idx[0].tolist())}

    # Fuse
    fused = reciprocal_rank_fusion([dense_ids, bm25_ids])

    q_vec = q_emb[0]
    q_norm = np.linalg.norm(q_vec) + 1e-9
    results = []
    for doc_id, rrf_score in fused:
        doc = documents[int(doc_id)]
        if filters:
            if filters.get("rating") and doc["metadata"]["rating"] != filters["rating"]:
                continue
            if filters.get("max_price"):
                price_val = float(re.sub(r"[^\d.]", "", doc["metadata"]["price"]) or 0)
                if price_val > filters["max_price"]:
                    continue
        d_vec = embeddings[int(doc_id)]
        cosine = float(np.dot(q_vec, d_vec) / (q_norm * (np.linalg.norm(d_vec) + 1e-9)))
        results.append({
            "chunk": doc["text"],
            "metadata": doc["metadata"],
            "hybrid_score": float(rrf_score),
            "similarity": cosine,
            "dense_score": dense_hits.get(doc_id),
            "bm25_score": bm25_hits.get(doc_id),
        })
        if len(results) >= top_k:
            break
    return results


def generate_answer(rewritten_query, docs):
    context = ""
    for d in docs:
        m = d["metadata"]
        context += (
            f"Title: {m['title']}\nPrice: {m['price']}\nRating: {m['rating']}\n"
            f"{d['chunk']}\n---------------------------------------\n"
        )
    prompt = f"""
You are an AI assistant for a book retrieval system.
Use ONLY the retrieved documents below to answer the user's question.

Retrieved Documents:
{context}

User Question:
{rewritten_query}

If the answer cannot be found, say:
'I could not find enough information in the retrieved documents.'
"""
    try:
        return gmodel.generate_content(prompt).text
    except Exception as e:
        return f"Gemini API error: {e}"


# ---------------------------------------------------------------------
# GUI
# ---------------------------------------------------------------------
st.title("📚 Book RAG — Hybrid Search")
st.caption("Dense embeddings + BM25 sparse retrieval, fused with Reciprocal Rank Fusion (RRF).")

with st.sidebar:
    st.header("Settings")
    top_k = st.slider("Chunks to retrieve", 1, 10, 5)
    st.markdown("**Retrieval:** Hybrid (dense + BM25 → RRF)")
    st.markdown(f"**Corpus size:** {len(documents)} books")

question = st.text_input("Your question", placeholder="e.g. cheap five star mystery books")
submitted = st.button("Submit", type="primary")

if submitted and question.strip():
    with st.spinner("Retrieving (hybrid) and generating answer..."):
        rewritten = rewrite_query(question)
        qclass = classify_query(rewritten)
        filters = extract_filters(rewritten)
        docs = hybrid_search(rewritten, top_k=top_k, filters=filters)
        answer = generate_answer(rewritten, docs)

    st.subheader("Final Answer")
    st.write(answer)

    st.subheader("Query Analysis")
    c1, c2 = st.columns(2)
    c1.markdown(f"**Original query**\n\n{question}")
    c2.markdown(f"**Rewritten query**\n\n{rewritten}")
    c3, c4 = st.columns(2)
    c3.markdown(f"**Query class**\n\n{qclass}")
    c4.markdown(f"**Extracted filters**\n\n`{filters}`")

    st.subheader(f"Retrieved Source Chunks ({len(docs)})")
    if not docs:
        st.info("No documents matched the extracted filters.")
    for i, d in enumerate(docs, start=1):
        m = d["metadata"]
        header = f"{i}. {m['title']}  —  hybrid {d['hybrid_score']:.4f} · cosine {d['similarity']:.3f}"
        with st.expander(header):
            s1, s2, s3 = st.columns(3)
            s1.metric("Hybrid (RRF)", f"{d['hybrid_score']:.4f}")
            s2.metric("Dense (cosine)", f"{d['similarity']:.3f}")
            s3.metric("BM25", "—" if d["bm25_score"] is None else f"{d['bm25_score']:.2f}")
            st.markdown(
                f"**Price:** {m['price']} &nbsp;|&nbsp; **Rating:** {m['rating']} "
                f"&nbsp;|&nbsp; [source]({m['source_url']})"
            )
            st.markdown("**Source metadata**")
            st.json(m)
            st.markdown("**Chunk text**")
            st.text(d["chunk"])
elif submitted:
    st.warning("Please type a question first.")


Writing app.py


In [78]:
# ---- Launch the Streamlit GUI from Google Colab ----
# Streamlit runs as a background process; localtunnel exposes it with a public URL.
# When you open the URL, localtunnel asks for a password = the IP printed below.

import time

# start Streamlit in the background (logs go to a file)
get_ipython().system_raw("streamlit run app.py --server.port 8501 &>/content/streamlit_logs.txt &")
time.sleep(6)

print("Tunnel password (paste this IP on the localtunnel page):")
get_ipython().system("wget -q -O - https://loca.lt/mytunnelpassword ; echo")
print("\nOpening tunnel — click the https://....loca.lt link below:\n")
get_ipython().system("npx --yes localtunnel --port 8501")

# ---------------------------------------------------------------------
# Alternative if localtunnel is unreliable — use pyngrok (needs a free token):
#   !pip install pyngrok -q
#   from pyngrok import ngrok
#   ngrok.set_auth_token("YOUR_NGROK_TOKEN")
#   get_ipython().system_raw("streamlit run app.py --server.port 8501 &>/content/streamlit_logs.txt &")
#   import time; time.sleep(6)
#   print("Public URL:", ngrok.connect(8501))
# ---------------------------------------------------------------------


Tunnel password (paste this IP on the localtunnel page):
34.125.47.164

Opening tunnel — click the https://....loca.lt link below:

⠙⠹⠸⠼⠴⠦⠧your url is: https://breezy-parrots-count.loca.lt


^C
